# Show T–S TEF Transects in the 79NG Fjord

Notebook by Markus Reinert (IOW, 2025–2026, https://orcid.org/0000-0002-3761-8029).

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
from cmocean import cm as cmo
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize
from shapely.geometry import MultiPoint
from shapely.plotting import plot_polygon

from tools.visualization import cm

from pyTEF.calc import calc_bulk_values

In [ ]:
# Define the linear equation of the freezing temperature used in GETM
# (in the figure, the result is indistinguishable from gsw.t_freezing)
l1 = -5.6705121472405570e-2
l2 =  7.5436448744204881e-2
l3 =  7.6828512903539831e-4
T_freezing = lambda S, z: l1 * S + l2 + l3 * z

In [ ]:
folder = Path("output")
!ls -lh $folder/TEF_lon_*.nc

## Open steady-state data

In [ ]:
ds = xr.open_dataset(folder / "out_mean_compressed.nc").squeeze(drop=True).set_coords("zc")

# Compute the mask
ds["mask"] = ds.bathymetry.notnull()
ds.mask.attrs = {"long_name": "ocean mask"}

# Check grid spacing
dlat = 1 / 240
dlon = 3 / 120
assert np.allclose(np.diff(ds.latc), dlat)
assert np.allclose(np.diff(ds.lonc), dlon)

# Update the attributes for better labels
ds.temp.attrs.update({"long_name": "temperature", "units": "°C"})
ds.salt.attrs.update({"long_name": "salinity", "units": "g/kg"})
ds.latc.attrs.update({"long_name": "latitude", "units": "°N"})
ds.lonc.attrs.update({"long_name": "longitude", "units": "°E"})

ds

## Open TEF analyses and compute bulk values

In [ ]:
q = {}
div_T = {}
div_S = {}

for i in [47, 107, 147]:
    filename = f"TEF_lon_{i:03d}_t207_s211.nc"
    with xr.open_dataset(folder / filename) as ds_tef:
        q[i] = ds_tef.q2
        Q = ds_tef.Q2
    q[i].plot(figsize=(6, 5))
    print(f"q-range at lon-index {i:3} ({ds.lonc[i]:.3f}{ds.lonc.units}):", end=" ")
    print(f"{q[i].min():10_.0f} to {q[i].max():10_.0f}")
    for dim, other in ["TS", "ST"]:
        print("Computing bulk values for", dim)
        bulk = calc_bulk_values(Q[dim], Q.mean(other))
        print(bulk)
        if dim == "T":
            # Store the second (or 3rd if more than 3) dividing temperature for later use
            div_T[i] = bulk.divval[1 if bulk.o.size == 3 else 2]
            line = plt.axhline
        else:
            # Store the second (middle) dividing salinity for later use
            div_S[i] = bulk.divval[1]
            line = plt.axvline
        for v in bulk.divval:
            line(v, ls="--", lw=0.5, color="gray")
    print("-" * 80)

q[i]

## Analyse the Gade slope of the mixing line

See Gade (1979) and Straneo & Cenedese (2015) for more details.

In [ ]:
# Define the effective ice temperature
L = 3.33e5  # J/kg
c_i = 1995  # J/kg/K
c_w = 3985  # J/kg/K
T_i = -20.  # °C
T_ice_effective = lambda S, z: T_freezing(S, z) - L / c_w - c_i / c_w * (T_freezing(S, z) - T_i)

# Compute the effective ice temperature for typical salinity and depth ranges at 79NG
S = np.linspace(0, 35, 71)
z = np.linspace(0, -600, 61)
T_ice_effective = T_ice_effective(S, z[:, np.newaxis])

# Show the effective ice temperature
plt.pcolormesh(S, z, T_ice_effective)
plt.colorbar()

T_ice_effective_min = T_ice_effective.min()
T_ice_effective_max = T_ice_effective.max()
print(f"The effective ice temperature goes from {T_ice_effective_min:.2f} °C to {T_ice_effective_max:.2f} °C.")

In [ ]:
for l in sorted(q):
    T = div_T[l]
    S = div_S[l]
    slope_min = (T - T_ice_effective_max) / S
    slope_max = (T - T_ice_effective_min) / S
    print(f"For the transect at index {l:3}, the dividing values are {T = :5.2f}, {S = :.2f},", end=" ")
    print(f"and the Gade slope is between {slope_min:.3f} and {slope_max:.3f}.")
    q[l].plot(figsize=(6, 5), robust=True)
    plt.axhline(T, ls=":")
    plt.axvline(S, ls=":")
    plt.plot(q[l].s, q[l].s * slope_min + T_ice_effective_max, "k", label="mixing line (min. Gade slope)")
    plt.plot(q[l].s, q[l].s * slope_max + T_ice_effective_min, "r--", label="mixing line (max. Gade slope)")
    plt.legend()

## Create the figure

In [ ]:
# Set the style of the plume labels
bbox = {"boxstyle": "square", "facecolor": "0.95", "alpha": 2/3, "edgecolor": "none"}
label_plume = lambda ax, text, coords: ax.text(*coords, text, size="x-small", bbox=bbox)

In [ ]:
fig, axs = plt.subplots(3, 2, width_ratios=[2, 3], constrained_layout=True, figsize=(18*cm, 18*cm), dpi=300)
fig.suptitle("Circulation in T–S coordinates at 3 transects across the 79NG fjord", weight="bold")

gray = "0.3"

for i, l in enumerate(sorted(q)):
    transect = ds.isel(lonc=l)
    # Select the first continuous part of ocean points in the transect with land in the south and north
    i_first_land = transect.mask.argmin().item()
    transect = transect.isel(latc=slice(i_first_land, None))
    i_first_water = transect.mask.argmax().item()
    transect = transect.isel(latc=slice(i_first_water, None))
    i_second_land = transect.mask.argmin().item()
    transect = transect.isel(latc=slice(i_second_land))

    ax = axs[i, 0]
    im = (q[l] / 1e6).plot(ax=ax, vmax=1, cmap=cmo.balance, add_colorbar=False)
    if i == 0:
        # Draw slightly extended convex hulls around the four different parts of the T–S diagram
        kwargs = dict(ax=ax, fc="none", ec=gray, ls=":", lw=0.75, add_points=False)
        # Mark the three subglacial plumes
        shallow = transect.zc > -300
        for s in [slice(79.42), slice(79.42, 79.58), slice(79.58, None)]:
            S = transect.sel(latc=s).salt.where(shallow).data.flatten()
            T = transect.sel(latc=s).temp.where(shallow).data.flatten()
            valid = np.isfinite(S)
            plot_polygon(MultiPoint(list(zip(S[valid], T[valid]))).convex_hull.buffer(0.04), **kwargs)
        # Mark the deeper flow
        S = transect.salt.where(~shallow).data.flatten()
        T = transect.temp.where(~shallow).data.flatten()
        valid = np.isfinite(S)
        plot_polygon(MultiPoint(list(zip(S[valid], T[valid]))).convex_hull.buffer(0.04), **kwargs)
        # Annotate the four parts
        ax.annotate("southern part", (33.55, -0.1), (32.9, 0.6), arrowprops={"arrowstyle": "->", "lw": 0.6, "edgecolor": gray}, size="x-small", color=gray)
        ax.text(33.45, -1.3, "central part", size="x-small", va="top", color=gray)
        ax.text(32.8, -0.1, "northern\npart", size="x-small", va="top", color=gray)
        ax.text(34.58, 1.5, "deep\npart", size="x-small", ha="center", va="top", color=gray)
        # Scatter plots used previously in the analysis but not for the final figure:
        # part = transect.sel(latc=slice(79.42, 79.58))
        # ax.scatter(part.salt, part.temp, 1, xr.ones_like(part.temp).where(part.zc > -300) * part.latc)
        text = "in the central cavity"
    elif i == 1:
        # Show and annotate the dividing salinity and temperature
        xmin, xmax = ax.get_xlim()
        dx = (xmax - xmin) / 100
        ymin, ymax = ax.get_ylim()
        dy = (ymax - ymin) / 100
        # colors for dividing values suggested by ChatGPT
        col_S = "#00B27A"
        col_T = "#FF8C00"
        ax.axvline(div_S[l], c=col_S, lw=1, ls=":")
        ax.axhline(div_T[l], c=col_T, lw=1, ls=":")
        ax.text(div_S[l] + dx, ymin + dy, f"{div_S[l]:.2f} {transect.salt.units}", size="small", c=col_S, va="bottom")
        ax.text(xmax - dx, div_T[l] - dy, f"{div_T[l]:.2f} {transect.temp.units}", size="small", c=col_T, va="top", ha="right")
        text = "near the main\ncalving front"
    else:
        # Show and annotate the freezing temperature at sea level
        T_freezing(q[l].s, 0).plot(ax=ax, ls=":", lw=1, color=gray)
        arrowprops = {"arrowstyle": "->", "lw": 0.6, "shrinkB": 4, "shrinkA": 0, "relpos": (0, 0.5), "color": gray}
        ax.annotate("freezing point\nat sea level", (33.3, T_freezing(33.3, 0)), (33.6, -1.7), size="small", color=gray, arrowprops=arrowprops)
        text = "at the fjord mouth"
        # Show and annotate the Gade slope
        slope = 2.8  # typical value from the computation above
        xrange = np.array([33.7, 34.3])  # custom salinity range
        yrange = slope * (xrange - xrange[0]) - 0.5  # custom temperature offset to ensure that the line does not obscure the data
        ax.plot(xrange, yrange, "--", color=gray, lw=1)
        ax.text(34.3, yrange[0] + np.diff(yrange) / 2, f"Gade slope\n{slope} °C (g/kg)$^{{-1}}$", size="small", va="top", ha="center", color=gray)
    ax.text(0.01, 0.96, f"({'ace'[i]}) {-transect.lonc:.1f}°W", transform=ax.transAxes, va="top")
    ax.text(0.11, 0.87, text, transform=ax.transAxes, va="top")
    if i == 2:
        ax.set_xlabel(f"{q[l].s.long_name} in {q[l].s.units}")
    else:
        ax.set_xlabel("")
        ax.xaxis.set_major_formatter("")
    ax.set_ylabel(f"{q[l].t.long_name} in {q[l].t.units}")

    ax = axs[i, 1]
    cs = transect.velx3d.plot.contourf(ax=ax, y="zc", levels=np.linspace(-0.15, 0.15, 101), cmap=cmo.balance, add_colorbar=False)
    if i == 0:
        # Show the divisions
        ax.plot([79.42, 79.42], [-100, -300], ":", lw=0.75, color=gray)
        ax.plot([79.58, 79.58], [-100, -300], ":", lw=0.75, color=gray)
        ax.plot([transect.latc[0] - 5*dlat, transect.latc[-1] + 5*dlat], [-300, -300], ":", lw=0.75, color=gray)
        ax.text(79.385, -240, "southern part", size="x-small", ha="right", color=gray)
        ax.text(transect.latc[-1] + 0.002, -240, "northern part", size="x-small", color=gray)
        ax.text(79.5, -140, "central part", size="x-small", ha="center", color=gray)
        ax.text(79.4, -420, "deep part", size="x-small", ha="right", va="top", color=gray)
        label_plume(ax, "p1", (79.395, -160))
        label_plume(ax, "p2", (79.465, -220))
        label_plume(ax, "p3", (79.570, -160))
    elif i == 1:
        # Show and annotate the dividing contours of both tracers
        for c, val, data, col, dy in [("S", div_S[l], transect.salt, col_S, -10), ("T", div_T[l], transect.temp, col_T, +10)]:
            cs = data.plot.contour(ax=ax, y="zc", levels=[val], colors=col, linewidths=1, linestyles=":")
            vertices = cs.get_paths()[0].vertices
            # Get the coordinates of the leftmost point of the contour line
            x_contour, y_contour = vertices[vertices.argmin(0)[0]]
            ax.annotate(
                f"${c} = {val:.2f}$ {data.units}",
                (x_contour, y_contour), (x_contour - 0.04, y_contour + dy),
                size="small", ha="right", va="top" if dy < 0 else "bottom", color=col,
                arrowprops={"arrowstyle": "-", "linewidth": 0.6, "relpos": (1, 0.5), "color": col},
            )
        label_plume(ax, "p1", (79.44, -50))
        # plume 2 is absent in this transect
        label_plume(ax, "p3", (79.59, -50))
        label_plume(ax, "AIW plume", (79.65, -400))
    ax.text(0.01, 0.96, f"({'bdf'[i]}) {-transect.lonc:.1f}°W", transform=ax.transAxes, va="top")
    if i == 2:
        ax.set_xlabel("latitude")
        ax.xaxis.set_major_formatter(lambda x, pos: f"{round(x, 10)}°N")
    else:
        ax.set_xlabel("")
        ax.xaxis.set_major_formatter("")
    ax.set(ylabel="depth in m", title="", xlim=(79.25, 79.74), ylim=(-883, 0))
    ax.yaxis.set_major_formatter(lambda x, pos: -int(x))

fig.colorbar(im, ax=axs[:, 0], location="top", pad=0.01, shrink=0.8, extend="both", ticks=[-1, 0, 1], aspect=20*2/3, label="zonal transport in Sv (°C)$^{-1}$ (g/kg)$^{-1}$")
# Create a continuous colorbar for the contours
sm = plt.cm.ScalarMappable(norm=Normalize(cs.cvalues.min(), cs.cvalues.max()), cmap=cmo.balance)
cbar = fig.colorbar(sm, ax=axs[:, 1], location="top", pad=0.01, shrink=0.8, extend="both", ticks=[-0.1, 0, 0.1], label="zonal velocity in m s$^{-1}$")
cbar.ax.text(-0.05, 0.485, "⊗", ha="center", va="center")
cbar.ax.text(+0.05, 0.485, "⊙", ha="center", va="center")

fig.savefig("figures/fig_8_TS_transports.png", dpi=600)